# FastAPI + SQLAlchemy Async

Demonstrates async SQLAlchemy patterns with SQLite in-memory (via `aiosqlite`). No external database server required.

```bash
pip install sqlalchemy aiosqlite fastapi httpx
```

In [ ]:
# !pip install sqlalchemy aiosqlite fastapi httpx

from sqlalchemy.ext.asyncio import create_async_engine, AsyncSession, async_sessionmaker
from sqlalchemy.orm import DeclarativeBase, mapped_column, Mapped
from sqlalchemy import Integer, String, select
import asyncio

# --- SQLite in-memory async engine ---
DATABASE_URL = "sqlite+aiosqlite:///:memory:"
engine = create_async_engine(DATABASE_URL, echo=False)
AsyncSessionLocal = async_sessionmaker(engine, expire_on_commit=False)

# --- ORM model with mapped_column (SQLAlchemy 2.x style) ---
class Base(DeclarativeBase):
    pass

class User(Base):
    __tablename__ = "users"

    id: Mapped[int] = mapped_column(Integer, primary_key=True, index=True)
    name: Mapped[str] = mapped_column(String(100), nullable=False)
    email: Mapped[str] = mapped_column(String(200), unique=True, nullable=False)
    age: Mapped[int] = mapped_column(Integer, nullable=True)

    def __repr__(self):
        return f"<User id={self.id} name={self.name!r} email={self.email!r}>"

# Create tables
async def init_db():
    async with engine.begin() as conn:
        await conn.run_sync(Base.metadata.create_all)
    print("Tables created.")

await init_db()

In [ ]:
# --- Basic CRUD: Create and Read ---

async def create_user(name: str, email: str, age: int = None) -> User:
    async with AsyncSessionLocal() as session:
        user = User(name=name, email=email, age=age)
        session.add(user)
        await session.commit()
        await session.refresh(user)
        return user

async def get_user(user_id: int) -> User | None:
    async with AsyncSessionLocal() as session:
        return await session.get(User, user_id)

async def get_all_users() -> list[User]:
    async with AsyncSessionLocal() as session:
        result = await session.execute(select(User))
        return result.scalars().all()

# Seed some users
u1 = await create_user("Alice", "alice@example.com", age=30)
u2 = await create_user("Bob", "bob@example.com", age=25)
u3 = await create_user("Charlie", "charlie@example.com", age=35)

print("Created:", u1, u2, u3)

user = await get_user(1)
print("Fetched by ID:", user)

all_users = await get_all_users()
print("All users:", all_users)

In [ ]:
# --- Query with filters and ordering ---
from sqlalchemy import asc, desc

async def get_users_filtered(min_age: int = None, order_by: str = "name") -> list[User]:
    async with AsyncSessionLocal() as session:
        stmt = select(User)
        if min_age is not None:
            stmt = stmt.where(User.age >= min_age)
        col = getattr(User, order_by, User.name)
        stmt = stmt.order_by(asc(col))
        result = await session.execute(stmt)
        return result.scalars().all()

# Filter: age >= 30, ordered by name
filtered = await get_users_filtered(min_age=30, order_by="name")
print("Age >= 30, ordered by name:")
for u in filtered:
    print(f"  {u.name}, age={u.age}")

# All users ordered by age descending
async with AsyncSessionLocal() as session:
    result = await session.execute(select(User).order_by(desc(User.age)))
    users_by_age = result.scalars().all()

print("\nAll users by age desc:")
for u in users_by_age:
    print(f"  {u.name}, age={u.age}")

In [ ]:
# --- Update with model_dump(exclude_unset=True) for PATCH ---
from pydantic import BaseModel
from typing import Optional

class UserUpdate(BaseModel):
    name: Optional[str] = None
    email: Optional[str] = None
    age: Optional[int] = None

async def patch_user(user_id: int, update_data: UserUpdate) -> User | None:
    async with AsyncSessionLocal() as session:
        user = await session.get(User, user_id)
        if not user:
            return None
        # Only update fields that were explicitly provided
        changes = update_data.model_dump(exclude_unset=True)
        print(f"  Applying changes: {changes}")
        for field, value in changes.items():
            setattr(user, field, value)
        await session.commit()
        await session.refresh(user)
        return user

# Patch only the age of user 1
patch = UserUpdate(age=31)
updated = await patch_user(1, patch)
print("After PATCH (age only):", updated)

# Patch name and email of user 2
patch2 = UserUpdate(name="Robert", email="robert@example.com")
updated2 = await patch_user(2, patch2)
print("After PATCH (name+email):", updated2)

In [ ]:
# --- Delete and full FastAPI integration ---
from fastapi import FastAPI, Depends, HTTPException
from fastapi.testclient import TestClient
from sqlalchemy.ext.asyncio import AsyncSession

app = FastAPI()

async def get_session():
    async with AsyncSessionLocal() as session:
        yield session

@app.delete("/users/{user_id}")
async def delete_user(user_id: int, session: AsyncSession = Depends(get_session)):
    user = await session.get(User, user_id)
    if not user:
        raise HTTPException(status_code=404, detail="User not found")
    await session.delete(user)
    await session.commit()
    return {"deleted": user_id}

@app.get("/users")
async def list_users(session: AsyncSession = Depends(get_session)):
    result = await session.execute(select(User).order_by(User.id))
    users = result.scalars().all()
    return [{"id": u.id, "name": u.name, "email": u.email, "age": u.age} for u in users]

client = TestClient(app)

# List all users
r = client.get("/users")
print("Users before delete:", r.json())

# Delete user 3
r = client.delete("/users/3")
print("Delete user 3:", r.json())

# List again
r = client.get("/users")
print("Users after delete:", r.json())

# Delete non-existent
r = client.delete("/users/99")
print("Delete missing:", r.status_code, r.json())